In [1]:
!pip install -q pandas scikit-learn

In [3]:
from google.colab import files
uploaded = files.upload()

Saving train_data.txt to train_data.txt
Saving test_data.txt to test_data.txt


In [4]:
import os
print("Training file exists:", os.path.exists("/content/train_data.txt"))
print("Testing file exists:", os.path.exists("/content/test_data.txt"))

Training file exists: True
Testing file exists: True


In [5]:
with open("/content/train_data.txt", "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline())

1 ::: Oscar et la dame rose (2009) ::: drama ::: Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.

2 ::: Cupid (1997) ::: thriller ::: A brother and sister with a past incestuous relationship have a current murderous relationship. He murders the women who reject him and she murders the women who get too close to him.

3 ::: Young, Wild and Wonderful (1980) ::: adult ::: As the bus empties the students for their field trip to the Museum of Natural History, little does the tour guide suspect that the

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Load training data
data = []

with open("/content/train_data.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(" ::: ", 3)
        if len(parts) == 4:
            movie_id, title, genre, description = parts
            data.append([title, genre, description])

df = pd.DataFrame(
    data, columns=["title", "genre", "description"]
)

print("Total movies:", len(df))
print("\nGenre distribution:")
print(df["genre"].value_counts())

# 2. Combine title and plot summary
df["text"] = df["title"] + " " + df["description"]

X = df["text"]
y = df["genre"]

# 3. Split data into training and testing portions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Convert text into TF-IDF features
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Train Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. Evaluate the model
predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(
    y_test, predictions, zero_division=0
))

Total movies: 54214

Genre distribution:
genre
drama          13613
documentary    13096
comedy          7447
short           5073
horror          2204
thriller        1591
action          1315
western         1032
reality-tv       884
family           784
adventure        775
music            731
romance          672
sci-fi           647
adult            590
crime            505
animation        498
sport            432
talk-show        391
fantasy          323
mystery          319
musical          277
biography        265
history          243
game-show        194
news             181
war              132
Name: count, dtype: int64

Model Accuracy: 46.87 %

Classification Report:
              precision    recall  f1-score   support

      action       0.00      0.00      0.00       263
       adult       0.00      0.00      0.00       118
   adventure       0.00      0.00      0.00       155
   animation       0.00      0.00      0.00       100
   biography       0.00      0.00      0

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    C=4,
    class_weight="balanced",
    solver="lbfgs"
)

lr_model.fit(X_train_tfidf, y_train)

# Predict genres
lr_predictions = lr_model.predict(X_test_tfidf)

# Evaluate
print("Logistic Regression Accuracy:",
      round(accuracy_score(y_test, lr_predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test, lr_predictions, zero_division=0
))

Logistic Regression Accuracy: 56.59 %

Classification Report:
              precision    recall  f1-score   support

      action       0.38      0.53      0.44       263
       adult       0.53      0.68      0.59       118
   adventure       0.30      0.36      0.33       155
   animation       0.25      0.25      0.25       100
   biography       0.07      0.02      0.03        53
      comedy       0.58      0.57      0.58      1490
       crime       0.16      0.17      0.16       101
 documentary       0.77      0.72      0.74      2619
       drama       0.68      0.53      0.60      2723
      family       0.23      0.30      0.26       157
     fantasy       0.19      0.17      0.18        65
   game-show       0.81      0.64      0.71        39
     history       0.18      0.16      0.17        49
      horror       0.62      0.71      0.67       441
       music       0.46      0.73      0.56       146
     musical       0.16      0.16      0.16        55
     mystery       

In [8]:
# Train the final TF-IDF vectorizer on all labelled data
X_all = df["text"]
y_all = df["genre"]

final_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_all_tfidf = final_vectorizer.fit_transform(X_all)

# Train Logistic Regression on all labelled data
final_model = LogisticRegression(
    max_iter=1000,
    C=4,
    class_weight="balanced",
    solver="lbfgs"
)

final_model.fit(X_all_tfidf, y_all)

print("Final model trained successfully!")

Final model trained successfully!


In [9]:
# Load test dataset
test_data = []

with open("/content/test_data.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(" ::: ", 2)
        if len(parts) == 3:
            movie_id, title, description = parts
            test_data.append([movie_id, title, description])

test_df = pd.DataFrame(
    test_data,
    columns=["id", "title", "description"]
)

# Convert text using the final vectorizer
test_df["text"] = (
    test_df["title"].fillna("") + " " +
    test_df["description"].fillna("")
)

test_tfidf = final_vectorizer.transform(test_df["text"])

# Predict genres
test_df["predicted_genre"] = final_model.predict(test_tfidf)

# Save predictions
test_df[["id", "title", "predicted_genre"]].to_csv(
    "/content/movie_genre_predictions.csv", index=False
)

print("Total predictions:", len(test_df))
print("\nSample predictions:")
print(test_df[["title", "predicted_genre"]].head(10).to_string(index=False))
print("\nPredictions saved successfully!")

Total predictions: 54200

Sample predictions:
                                     title predicted_genre
                      Edgar's Lunch (1998)           short
                  La guerra de papá (1977)           drama
               Off the Beaten Track (2010)     documentary
                    Meu Amigo Hindu (2015)           drama
                         Er nu zhai (1955)           drama
                        Riddle Room (2016)        thriller
                            L'amica (1969)           drama
                      Ina Mina Dika (1989)          comedy
Equinox Special: Britain's Tornados (2005)     documentary
                              Press (2011)           short

Predictions saved successfully!


In [10]:
import joblib

joblib.dump(
    {
        "vectorizer": final_vectorizer,
        "model": final_model
    },
    "/content/movie_genre_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [11]:
from google.colab import files

files.download("/content/movie_genre_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
files.download("/content/movie_genre_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>